### Homework: Multimodal Models, Vision Transformers, and PerceiverIO

#### Instructions:
- Complete the following exercises based on the lecture material.
- Submit your completed notebook with all cells executed.

---

## Part 1: Introduction to Multimodal Models

### Question 1: Understanding Multimodal Learning
**(a)** What are multimodal models, and how do they differ from unimodal models?

- Multimodal models are capable of combining information from multiple input modalities while unimodal models use a single input modality such as text
- Multimodal models align information from each modality for reasoning across modalities
- Some multimodal models use separate encoders for each modality to map features into a shared latent space

**(b)** List three real-world applications where multimodal learning significantly improves performance.

- Speech-to-text & translation models such as Whisper combine audio and text modalities
- Image QA, captioning, and reasoning tasks with models such as gpt-4o combine image and text modalities
- Weather prediction models combine information from different geospatial data sources

### Question 2: Implementing a Simple Multimodal Fusion Model
Modify the given function to combine image and text features into a joint representation using a simple feed-forward network.

In [3]:
import torch
import torch.nn as nn

class MultimodalFusion(nn.Module):
    def __init__(self, image_dim, text_dim, fusion_dim):
        super().__init__()
        self.fc_image = nn.Linear(image_dim, fusion_dim)
        self.fc_text = nn.Linear(text_dim, fusion_dim)
        # per modality normalization layers
        self.fc_image_norm = nn.LayerNorm(fusion_dim)
        self.fc_text_norm = nn.LayerNorm(fusion_dim)
        self.fc_fusion = nn.Linear(fusion_dim, fusion_dim)

    def forward(self, img_feat, text_feat):
        img_proj = self.fc_image(img_feat)
        text_proj = self.fc_text(text_feat)
        # normalize projections before fusion
        fused = torch.relu(self.fc_image_norm(img_proj) + self.fc_text_norm(text_proj))
        return self.fc_fusion(fused)

# Test with dummy input
image_feat = torch.rand(32, 512)
text_feat = torch.rand(32, 512)
fusion_model = MultimodalFusion(512, 512, 256)
out = fusion_model(image_feat, text_feat)
print(out.shape)

torch.Size([32, 256])


Modify the function to include a normalization layer before fusion.

---

## Part 2: Vision Transformer and CLIP

### Question 3: Understanding Vision Transformers
**(a)** Explain how Vision Transformers (ViTs) differ from traditional Convolutional Neural Networks (CNNs) for image processing.

- Vision Transformers process the entire image at once, capturing local and global relationships between fixed size patches of the image

**(b)** What is the role of positional embeddings in ViTs?

- Positional embeddings inform the model of the spatial relationship between patches. Without positional embeddings, it would not be possible to model this type of relationship because the transformer doesn't have its own concept of positioning.

### Question 4: Experimenting with CLIP
Modify the following script to load CLIP and use it to classify an image based on text descriptions.

In [5]:
pip install openai-clip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
  Created wheel for openai-clip: filename=openai_clip-1.0.1-py3-none-any.whl size=1368605 sha256=07391749d4e8a99bc1bae903fc15fb12d97890f9cfe6865fc7ca744206092f15
  Stored in directory: /root/.cache/pip/wheels/ab/49/bc/c2342e8e14878210ba4825cf314a53f2570f6fb18b91fce3cf
Successfully built openai-clip


In [13]:
import torch
import clip
from PIL import Image

model, preprocess = clip.load("ViT-B/32", device="cuda" if torch.cuda.is_available() else "cpu")
images = [("cat.jpg", 2), ("dog.jpg", 1), ("horse.jpg", 3), ("bear.jpg", -1)]
text_inputs = clip.tokenize(["a goblin", "a dog", "a cat", "a horse"])
text_features = model.encode_text(text_inputs)

for (image, label) in images:
  image_input = preprocess(Image.open(image)).unsqueeze(0)
  with torch.no_grad():
    image_features = model.encode_image(image_input)
    similarity = (image_features @ text_features.T).softmax(dim=-1)

  predicted = similarity.argmax().item()
  print("Predicted label:", predicted, "for", image.replace(".jpg", ""), "correct" if label == predicted else "incorrect")

Predicted label: 2 for cat correct
Predicted label: 1 for dog correct
Predicted label: 3 for horse correct
Predicted label: 1 for bear incorrect


Modify the script to classify multiple images in a batch.

---

## Part 3: PerceiverIO and Generalized Multimodal Models

### Question 5: Understanding PerceiverIO
**(a)** What problem does PerceiverIO solve compared to Transformers and CNNs?

- PerceiverIO works with a wide range of modalities without customization. High bandwidth modalities such as image and audio are compressed into the same space as text and other modalities
- The size of the model is decoupled from input sizes and quadratic scaling for long inputs is eliminated

**(b)** Explain the concept of latent arrays in PerceiverIO and why they are useful.

- Latent array are a fixed size set of learned vectors which allow information from different input modalities to be mapped into the same space.
- Latent arrays are useful because the size of the input is decoupled from the model depth and compute requirements and because information from multiple modalities can be integrated in the same internal representation.

### Question 6: Applying PerceiverIO to Multimodal Data
Write a function that initializes a PerceiverIO model and processes both image and text data.


In [1]:
from transformers import PerceiverModel, PerceiverConfig
import torch

config = PerceiverConfig()
config.d_model = 512
model = PerceiverModel(config)

def process_multimodal_data(image_tensor, text_tensor, model, use_image=True, use_text=True):
    inputs = torch.stack([image_tensor, text_tensor], dim=1)

    # create an attention mask depending on selected modalities
    attention_mask = torch.tensor(
        [[1 if use_image else 0, 1 if use_text else 0]],
        dtype=torch.long
    ).repeat(inputs.size(0), 1)

    outputs = model(inputs, attention_mask=attention_mask)
    return outputs

# Test with dummy input
image_tensor = torch.rand(1, 512)
text_tensor = torch.rand(1, 512)

out_both = process_multimodal_data(image_tensor, text_tensor, model, use_image=True, use_text=True)
print("both:", out_both.last_hidden_state.shape)

out_img_only = process_multimodal_data(image_tensor, text_tensor, model, use_image=True, use_text=False)
print("img only:", out_img_only.last_hidden_state.shape)

out_txt_only = process_multimodal_data(image_tensor, text_tensor, model, use_image=False, use_text=True)
print("txt only:", out_txt_only.last_hidden_state.shape)

both: torch.Size([1, 256, 1280])
img only: torch.Size([1, 256, 1280])
txt only: torch.Size([1, 256, 1280])


Modify the function to include an attention mask for selective processing of modalities.

---

### Submission
Submit your completed notebook with answers and executed code outputs. Ensure that all model outputs, loss trends, and dataset analyses are included in your submission.